# BLIP Image Text Matching

Fresh Python 3.11+ CPU kernel. Downloads the exact official Salesforce BLIP checkpoint revision and original source image; no API key. Pins Transformers 4.x for the source APIs. Replace missing local checkpoint paths, use evaluation/inference mode, and independently recompute the model for each description instead of printing the same score three times. Original code and original historical outputs remain in appendices; current outputs are computed, not copied. Binary matching scores are not calibrated factual certainty.

In [ ]:
import os
import subprocess
import sys
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--disable-pip-version-check", "torch==2.10.0", "transformers==4.57.6", "numpy==2.4.6", "Pillow==13.0.0", "sentencepiece==0.2.1", "certifi==2026.10.23", "ipywidgets==8.1.7"], check=True)
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
import certifi
os.environ["REQUESTS_CA_BUNDLE"] = certifi.where()
import numpy as np
import torch
import transformers
from PIL import Image
torch.set_num_threads(2)
torch.manual_seed(42)
print("Python:", sys.version.split()[0], "| PyTorch:", torch.__version__, "| Transformers:", transformers.__version__)

## Example 0

In [ ]:
print("Pinned PyTorch and Transformers dependencies installed in the active notebook kernel.")

## Example 1

In [ ]:
from transformers import BlipForImageTextRetrieval
MODEL_ID = "Salesforce/blip-itm-base-coco"
REVISION = "bed8ad38cb2d04a5a4bdf2d071b3c3c0a4aa724c"
model = BlipForImageTextRetrieval.from_pretrained(MODEL_ID, revision=REVISION, token=False).to("cpu").eval()
print("Loaded:", MODEL_ID, "| revision:", REVISION)
print("Parameters:", sum(p.numel() for p in model.parameters()))

## Example 2

In [ ]:
from transformers import AutoProcessor
processor = AutoProcessor.from_pretrained(MODEL_ID, revision=REVISION, token=False, use_fast=False)
print("Processor:", type(processor).__name__)

## Example 3

In [ ]:
import io
import hashlib
import requests
img_url = "https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcTzURjiEwhDZgRX8n6pgxlm2Pf0LZ6HIdQUIg&s"
response = requests.get(img_url, timeout=60)
response.raise_for_status()
image_bytes = response.content
assert hashlib.sha256(image_bytes).hexdigest() == "82bc58986a4e416b4dc775a6b3aed18b18e4eb90df2d4769f3de80c1eec036a4", "The original remote image changed; review the new bytes before using them."
raw_image = Image.open(io.BytesIO(image_bytes)).convert("RGB")
print("Input image:", raw_image.size, raw_image.mode)
print("Image SHA-256:", hashlib.sha256(image_bytes).hexdigest())
display(raw_image)

## Example 4

In [ ]:
text_1 = "Solidarity demonstrations for Palestine"
text_2 = "Solidarity demonstrations for Israel"
text_3 = "A man plays with his dog on the beach"

texts = [text_1, text_2, text_3]
print("Candidate descriptions:")
for index, text in enumerate(texts, 1):
    print(index, text)

## Example 5

In [ ]:
inputs = processor(images=raw_image, text=text_1, return_tensors="pt")
print("Processor keys:", list(inputs))

## Example 6

In [ ]:
for key, value in inputs.items():
    print(key, tuple(value.shape), value.dtype)
print("Input token IDs:", inputs["input_ids"].tolist())
print("Attention mask:", inputs["attention_mask"].tolist())

## Example 7

In [ ]:
with torch.inference_mode():
    itm_scores = model(**inputs, use_itm_head=True).itm_score
    itm_score = torch.softmax(itm_scores, dim=1)
print("Not matched / matched:")
print(itm_score)

## Example 8

In [ ]:
scores = [float(itm_score[0, 1])]
print(f"The image and text_1 are matched with a probability of {scores[0]:.4f}")

## Example 9

In [ ]:
inputs_2 = processor(images=raw_image, text=text_2, return_tensors="pt")
with torch.inference_mode():
    score_2 = torch.softmax(model(**inputs_2, use_itm_head=True).itm_score, dim=1)
scores.append(float(score_2[0, 1]))
print(f"The image and text_2 are matched with a probability of {scores[1]:.4f}")

## Example 10

In [ ]:
inputs_3 = processor(images=raw_image, text=text_3, return_tensors="pt")
with torch.inference_mode():
    score_3 = torch.softmax(model(**inputs_3, use_itm_head=True).itm_score, dim=1)
scores.append(float(score_3[0, 1]))
print(f"The image and text_3 are matched with a probability of {scores[2]:.4f}")

## Independent regression checks

In [ ]:
assert not model.training
assert tuple(inputs["pixel_values"].shape) == (1, 3, 384, 384)
assert inputs["input_ids"].shape == inputs["attention_mask"].shape
assert len(scores) == 3 and np.isfinite(scores).all() and all(0 <= p <= 1 for p in scores)
batch = processor(images=[raw_image] * 3, text=texts, padding=True, return_tensors="pt")
with torch.inference_mode():
    logits = model(**batch, use_itm_head=True).itm_score
logits_np = logits.numpy().astype("float64")
exps = np.exp(logits_np - logits_np.max(axis=1, keepdims=True))
reference = exps / exps.sum(axis=1, keepdims=True)
np.testing.assert_allclose(reference.sum(axis=1), 1, atol=1e-12)
np.testing.assert_allclose(reference[:, 1], scores, atol=2e-6, rtol=1e-5)
reverse = processor(images=[raw_image] * 3, text=texts[::-1], padding=True, return_tensors="pt")
with torch.inference_mode():
    reverse_scores = torch.softmax(model(**reverse, use_itm_head=True).itm_score, dim=1)[:, 1].numpy()
np.testing.assert_allclose(reverse_scores[::-1], scores, atol=2e-6, rtol=1e-5)
assert all(p.grad is None for p in model.parameters())
print("All three actual model scores:", [round(p, 6) for p in scores])
print("Processor shapes, bounds, independent NumPy softmax, singleton/batch agreement, permutation and inference-mode checks passed.")

## Original source appendix

Full original blocks from raw source SHA-256 263b45d93e8c3d3e20ff75b67f1a5b47fac27734ddebe145c0e871dc5d1c3ebc. Preserved verbatim as nonexecuting historical material, including mistakes/legacy setup alternatives.

### Original block 1

```text
!pip install transformers
 !pip install torch
```

### Original block 2

```text
from transformers import BlipForImageTextRetrieval
model = BlipForImageTextRetrieval.from_pretrained(
    "./models/Salesforce/blip-itm-base-coco")
```

### Original block 3

```text
from transformers import AutoProcessor
processor = AutoProcessor.from_pretrained(
    "./models/Salesforce/blip-itm-base-coco")
```

### Original block 4

```text
from PIL import Image
import requests

img_url = 'https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcTzURjiEwhDZgRX8n6pgxlm2Pf0LZ6HIdQUIg&s'
raw_image =  Image.open(
    requests.get(img_url, stream=True).raw).convert('RGB')
raw_image
```

### Original block 5

```text
text_1 = "Solidarity demonstrations for Palestine"
text_2 = "Solidarity demonstrations for Israel"
text_3 = "A man plays with his dog on the beach"
```

### Original block 6

```text
inputs = processor(images=raw_image,
                   text=text_1,
                   return_tensors="pt")
```

### Original block 7

```text
inputs
```

### Original block 8

```text
itm_scores = model(**inputs)[0]
itm_score = torch.nn.functional.softmax(
    itm_scores,dim=1)
itm_score
```

### Original block 9

```text
print(f"""\
The image and text_1 are matched \
with a probability of {itm_score[0][1]:.4f}""")
```

### Original block 10

```text
print(f"""\
The image and text_2 are matched \
with a probability of {itm_score[0][1]:.4f}""")
```

### Original block 11

```text
print(f"""\
The image and text_3 are matched \
with a probability of {itm_score[0][1]:.4f}""")
```

## Original source outputs

{‘pixel_values’: tensor([[[[-3.6162e-01, -8.4247e-02, 2.2232e-01, …, 1.5508e+00, 1.5508e+00, 1.5654e+00], [-2.7403e-01, -2.5853e-02, 2.8071e-01, …, 1.6092e+00, 1.6092e+00, 1.6238e+00], [-1.1344e-01, 1.3473e-01, 4.2670e-01, …, 1.7406e+00, 1.7406e+00, 1.7406e+00], …, [-1.2229e+00, -1.1061e+00, -1.1937e+00, …, -1.2959e+00, -1.3689e+00, -1.3397e+00], [-1.3689e+00, -8.2877e-01, -7.2658e-01, …, -1.5149e+00, -1.3689e+00, -1.4127e+00], [-1.4273e+00, -7.1198e-01, -5.3680e-01, …, -1.6025e+00, -1.3689e+00, -1.4419e+00]],

[[-2.8134e-01, 3.8118e-03, 3.3398e-01, …, -3.1135e-01, -3.1135e-01, -2.9634e-01], [-1.9129e-01, 6.3843e-02, 3.9401e-01, …, -2.5132e-01, -2.5132e-01, -2.3631e-01], [-2.6204e-02, 2.1392e-01, 5.2908e-01, …, -1.1625e-01, -1.1625e-01, -1.0124e-01], …, [-1.2118e+00, -1.1368e+00, -1.2718e+00, …, -1.1968e+00, -1.2568e+00, -1.2418e+00], [-1.3469e+00, -8.3662e-01, -8.0661e-01, …, -1.4219e+00, -1.2568e+00, -1.3169e+00], [-1.4069e+00, -7.1656e-01, -6.1151e-01, …, -1.5120e+00, -1.2568e+00, -1.3469e+00]],

[[-8.6653e-02, 1.8353e-01, 4.8215e-01, …, -5.8213e-02, -4.3993e-02, -2.9773e-02], [-1.3329e-03, 2.4041e-01, 5.3903e-01, …, -1.3329e-03, 1.2887e-02, 2.7107e-02], [ 1.5509e-01, 3.9683e-01, 6.8123e-01, …, 1.2665e-01, 1.4087e-01, 1.4087e-01], …, [-1.0678e+00, -1.0110e+00, -1.1105e+00, …, -9.2564e-01, -9.5408e-01, -9.2564e-01], [-1.1958e+00, -7.2656e-01, -6.8390e-01, …, -1.1389e+00, -9.5408e-01, -9.9674e-01], [-1.2527e+00, -6.1280e-01, -5.1326e-01, …, -1.2243e+00, -9.5408e-01, -1.0252e+00]]]]), ‘input_ids’: tensor([[ 101, 14657, 13616, 2005, 8976, 102]]), ‘attention_mask’: tensor([[1, 1, 1, 1, 1, 1]])}

tensor([[0.0031, 0.9969]], grad_fn=<SoftmaxBackward0>)

The image and text_1 are matched with a probability of 0.9969

The image and text are matched with a probability of 0.7179

The image and text are matched with a probability of 0.0000